# DCN-v2 Training Pipeline (Phase 8.6)

**Purpose**: Train Deep & Cross Network v2 for credit risk assessment.

**Architecture**:
- 253 numerical features → SoftBinning → EmbeddingEngine → x₀
- x₀ → CrossNetwork (explicit interactions) + DeepNetwork (implicit) → concat → P(default)

**Protocol**: Pair-programming approved (2026-02-15)

---

## Cell 1: Setup & Imports

Mount Google Drive and import required libraries.

In [ ]:
# Purpose: Mount Google Drive and set up environment
# Why: Data and model checkpoints stored on Drive for persistence
# Approved: 2026-02-15

from google.colab import drive
drive.mount('/content/drive')

# Set project path (adjust to your Drive folder)
PROJECT_PATH = '/content/drive/MyDrive/final_ex'  # <-- EDIT THIS

import sys
sys.path.insert(0, f'{PROJECT_PATH}/src')

In [ ]:
# Purpose: Import all required libraries
# Why: Standard ML/DL stack plus DCN-v2 modules
# Approved: 2026-02-15

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import pandas as pd
import numpy as np
from sklearn.metrics import roc_auc_score
from typing import Dict, List, Tuple
import os

# DCN-v2 modules
from models.dcn_v2 import DCNv2, compute_quantiles_for_binning

# Device configuration
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

## Cell 2: Data Loading

Load preprocessed parquet files and create PyTorch DataLoader.

In [ ]:
# Purpose: Define paths to preprocessed data
# Why: Data was preprocessed in Phase 5.6 with target encoding (253 numerical features)
# Approved: 2026-02-15

DATA_PATH = f'{PROJECT_PATH}/data/processed'

# Load DataFrames
X_train = pd.read_parquet(f'{DATA_PATH}/vime_X_train_fixed.parquet')
X_val = pd.read_parquet(f'{DATA_PATH}/vime_X_val_fixed.parquet')
X_test = pd.read_parquet(f'{DATA_PATH}/vime_X_test_fixed.parquet')

y_train = pd.read_parquet(f'{DATA_PATH}/vime_y_train.parquet')
y_val = pd.read_parquet(f'{DATA_PATH}/vime_y_val.parquet')
y_test = pd.read_parquet(f'{DATA_PATH}/vime_y_test.parquet')

# Feature list (all 253 numerical)
NUMERICAL_FEATURES = X_train.columns.tolist()

print(f'Train: {X_train.shape}, Val: {X_val.shape}, Test: {X_test.shape}')
print(f'Features: {len(NUMERICAL_FEATURES)}')

In [ ]:
# Purpose: Create PyTorch Dataset for DCN-v2
# Why: DCN-v2 expects Dict[str, Tensor] input, not a single tensor
# Approved: 2026-02-15

class DCNv2Dataset(Dataset):
    """
    Dataset for DCN-v2 that provides Dict-based feature access.
    
    Each __getitem__ returns:
        numerical_values: Dict[str, float] - one value per feature
        label: float - 0 or 1
    """
    
    def __init__(self, X_df: pd.DataFrame, y_df: pd.DataFrame, feature_names: List[str]):
        # Store as numpy arrays for fast access
        self.X = X_df.values.astype(np.float32)
        self.y = y_df.values.astype(np.float32).flatten()
        self.feature_names = feature_names
        
    def __len__(self):
        return len(self.y)
    
    def __getitem__(self, idx: int) -> Tuple[Dict[str, torch.Tensor], torch.Tensor]:
        # Create dict of feature name -> value
        row = self.X[idx]
        numerical_values = {name: torch.tensor(row[i]) for i, name in enumerate(self.feature_names)}
        label = torch.tensor(self.y[idx])
        return numerical_values, label

In [ ]:
# Purpose: Custom collate function for Dict-based batching
# Why: Default collate doesn't handle Dict[str, Tensor] inputs properly
# Approved: 2026-02-15

def dcnv2_collate_fn(batch: List[Tuple[Dict[str, torch.Tensor], torch.Tensor]]):
    """
    Collate function that stacks Dict values into batched tensors.
    
    Input: List of (numerical_dict, label) tuples
    Output: (batched_numerical_dict, batched_labels)
    """
    numerical_dicts, labels = zip(*batch)
    
    # Get feature names from first item
    feature_names = numerical_dicts[0].keys()
    
    # Stack each feature across batch
    batched_numerical = {
        name: torch.stack([d[name] for d in numerical_dicts])
        for name in feature_names
    }
    
    batched_labels = torch.stack(labels).unsqueeze(1)  # Shape: (batch, 1)
    
    return batched_numerical, batched_labels

In [ ]:
# Purpose: Create DataLoaders with custom collate
# Why: Batch processing for efficient GPU training
# Approved: 2026-02-15

BATCH_SIZE = 1024

train_dataset = DCNv2Dataset(X_train, y_train, NUMERICAL_FEATURES)
val_dataset = DCNv2Dataset(X_val, y_val, NUMERICAL_FEATURES)
test_dataset = DCNv2Dataset(X_test, y_test, NUMERICAL_FEATURES)

train_loader = DataLoader(
    train_dataset, 
    batch_size=BATCH_SIZE, 
    shuffle=True,
    collate_fn=dcnv2_collate_fn,
    num_workers=2
)

val_loader = DataLoader(
    val_dataset, 
    batch_size=BATCH_SIZE, 
    shuffle=False,
    collate_fn=dcnv2_collate_fn,
    num_workers=2
)

test_loader = DataLoader(
    test_dataset, 
    batch_size=BATCH_SIZE, 
    shuffle=False,
    collate_fn=dcnv2_collate_fn,
    num_workers=2
)

print(f'Train batches: {len(train_loader)}, Val batches: {len(val_loader)}, Test batches: {len(test_loader)}')

## Cell 3: Quantile Computation

Compute quantiles for SoftBinning initialization.

In [ ]:
# Purpose: Compute quantiles for each numerical feature
# Why: SoftBinning needs cutpoint initialization from training data
# Approved: 2026-02-15

NUM_BINS = 10

# Compute quantiles for all features
quantiles_dict = {}
for feat in NUMERICAL_FEATURES:
    values = X_train[feat].values
    quantiles = compute_quantiles_for_binning(torch.tensor(values), NUM_BINS)
    quantiles_dict[feat] = quantiles

print(f'Computed quantiles for {len(quantiles_dict)} features')
print(f'Sample quantiles for {NUMERICAL_FEATURES[0]}:')
print(quantiles_dict[NUMERICAL_FEATURES[0]])

## Cell 4: Model Instantiation

Create DCN-v2 model and print architecture summary.

In [ ]:
# Purpose: Instantiate DCN-v2 model
# Why: All 253 features treated as numerical (target-encoded data)
# Approved: 2026-02-15

model = DCNv2(
    numerical_features=NUMERICAL_FEATURES,
    categorical_cardinalities={},  # No categoricals (all target-encoded)
    num_bins=NUM_BINS,
    numerical_embed_dim=16,
    cross_layers=3,
    cross_rank=64,
    deep_hidden=[512, 256, 128],
    dropout=0.1
)

model = model.to(device)

# Count parameters
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f'Total parameters: {total_params:,}')
print(f'Trainable parameters: {trainable_params:,}')
print(f'Model output dimension: {model.embedding.output_dim}')

In [ ]:
# Purpose: Initialize SoftBinning cutpoints with computed quantiles
# Why: Data-driven initialization for better convergence
# Approved: 2026-02-15

# Access each SoftBinning module and set cutpoints
for feat in NUMERICAL_FEATURES:
    soft_bin = model.embedding.numerical_binning[feat]
    soft_bin.cutpoints.data = quantiles_dict[feat].to(device)

print('SoftBinning cutpoints initialized from training quantiles')

## Cell 5: Training Loop

Train with class-weighted BCE loss, early stopping, and checkpointing.

In [ ]:
# Purpose: Set up loss function and optimizer
# Why: Class imbalance (~8% default rate) requires pos_weight
# Approved: 2026-02-15

# Compute class weight from training data
n_positive = y_train.values.sum()
n_negative = len(y_train) - n_positive
pos_weight = torch.tensor([n_negative / n_positive], device=device)

print(f'Class distribution: {n_positive:.0f} positive ({100*n_positive/len(y_train):.1f}%), {n_negative:.0f} negative')
print(f'pos_weight: {pos_weight.item():.2f}')

# BCE with logits (more numerically stable)
# We'll modify forward to return logits instead of sigmoid for training
criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)

# AdamW optimizer with weight decay
optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=1e-3,
    weight_decay=1e-5
)

In [ ]:
# Purpose: Training and evaluation functions
# Why: Modular code for clean training loop
# Approved: 2026-02-15

def train_epoch(model, loader, criterion, optimizer, device):
    """
    Train for one epoch.
    
    Returns:
        avg_loss: float
    """
    model.train()
    total_loss = 0.0
    n_batches = 0
    
    for numerical_values, labels in loader:
        # Move to device
        numerical_values = {k: v.to(device) for k, v in numerical_values.items()}
        labels = labels.to(device)
        
        # Forward pass (model returns sigmoid, but we need logits for BCEWithLogitsLoss)
        # Temporarily access internal components
        x0 = model.embedding(numerical_values, {})
        cross_out = model.cross_network(x0)
        deep_out = model.deep_network(x0)
        combined = torch.cat([cross_out, deep_out], dim=1)
        logits = model.head(combined)
        
        # Compute loss
        loss = criterion(logits, labels)
        
        # Backward pass
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        
        total_loss += loss.item()
        n_batches += 1
    
    return total_loss / n_batches


def evaluate(model, loader, device):
    """
    Evaluate model and compute AUC-ROC.
    
    Returns:
        auc: float
    """
    model.eval()
    all_preds = []
    all_labels = []
    
    with torch.no_grad():
        for numerical_values, labels in loader:
            # Move to device
            numerical_values = {k: v.to(device) for k, v in numerical_values.items()}
            
            # Forward pass (returns sigmoid probabilities)
            probs = model(numerical_values, {})
            
            all_preds.extend(probs.cpu().numpy().flatten())
            all_labels.extend(labels.numpy().flatten())
    
    auc = roc_auc_score(all_labels, all_preds)
    return auc

In [ ]:
# Purpose: Main training loop with early stopping
# Why: Prevent overfitting by monitoring validation AUC
# Approved: 2026-02-15

N_EPOCHS = 100
PATIENCE = 10
CHECKPOINT_PATH = f'{PROJECT_PATH}/outputs/models/dcn_v2_best.pth'

# Create output directory if needed
os.makedirs(os.path.dirname(CHECKPOINT_PATH), exist_ok=True)

best_val_auc = 0.0
patience_counter = 0
history = {'train_loss': [], 'val_auc': []}

for epoch in range(N_EPOCHS):
    # Train
    train_loss = train_epoch(model, train_loader, criterion, optimizer, device)
    
    # Evaluate
    val_auc = evaluate(model, val_loader, device)
    
    # Log
    history['train_loss'].append(train_loss)
    history['val_auc'].append(val_auc)
    
    print(f'Epoch {epoch+1:3d}/{N_EPOCHS} | Train Loss: {train_loss:.4f} | Val AUC: {val_auc:.4f}')
    
    # Checkpointing
    if val_auc > best_val_auc:
        best_val_auc = val_auc
        patience_counter = 0
        torch.save({
            'epoch': epoch,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'val_auc': val_auc,
        }, CHECKPOINT_PATH)
        print(f'  --> New best! Saved checkpoint.')
    else:
        patience_counter += 1
        if patience_counter >= PATIENCE:
            print(f'\nEarly stopping at epoch {epoch+1} (no improvement for {PATIENCE} epochs)')
            break

print(f'\nBest validation AUC: {best_val_auc:.4f}')

## Cell 6: Evaluation

Load best model and evaluate on test set.

In [ ]:
# Purpose: Load best checkpoint and evaluate on test set
# Why: Final evaluation uses best model by validation AUC
# Approved: 2026-02-15

# Load best model
checkpoint = torch.load(CHECKPOINT_PATH)
model.load_state_dict(checkpoint['model_state_dict'])
print(f'Loaded best model from epoch {checkpoint["epoch"]+1}')

# Evaluate on test set
test_auc = evaluate(model, test_loader, device)
print(f'\nTest AUC: {test_auc:.4f}')

# Compare to baselines
print('\n--- Comparison to Previous Routes ---')
print(f'Route A (VIME + CatBoost):     0.6912')
print(f'Route B (Raw + CatBoost):      0.7391')
print(f'Route F (Two-Model Ensemble):  0.7369')
print(f'DCN-v2:                        {test_auc:.4f}')

In [ ]:
# Purpose: Plot training history
# Why: Visual inspection of training dynamics
# Approved: 2026-02-15

import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Training loss
axes[0].plot(history['train_loss'])
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Training Loss')
axes[0].set_title('Training Loss over Epochs')
axes[0].grid(True)

# Validation AUC
axes[1].plot(history['val_auc'])
axes[1].axhline(y=best_val_auc, color='r', linestyle='--', label=f'Best: {best_val_auc:.4f}')
axes[1].axhline(y=0.7391, color='g', linestyle='--', label='Route B: 0.7391')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Validation AUC')
axes[1].set_title('Validation AUC over Epochs')
axes[1].legend()
axes[1].grid(True)

plt.tight_layout()
plt.savefig(f'{PROJECT_PATH}/outputs/figures/dcn_v2_training_history.png', dpi=150)
plt.show()

In [ ]:
# Purpose: Save training results
# Why: Reproducibility and comparison with other routes
# Approved: 2026-02-15

import json

results = {
    'model': 'DCN-v2',
    'best_epoch': checkpoint['epoch'] + 1,
    'val_auc': best_val_auc,
    'test_auc': test_auc,
    'n_parameters': total_params,
    'hyperparameters': {
        'num_bins': NUM_BINS,
        'numerical_embed_dim': 16,
        'cross_layers': 3,
        'cross_rank': 64,
        'deep_hidden': [512, 256, 128],
        'dropout': 0.1,
        'batch_size': BATCH_SIZE,
        'lr': 1e-3,
        'weight_decay': 1e-5,
        'patience': PATIENCE
    }
}

results_path = f'{PROJECT_PATH}/outputs/results/dcn_v2_results.json'
with open(results_path, 'w') as f:
    json.dump(results, f, indent=2)

print(f'Results saved to {results_path}')

---

## Next Steps

After training completes:
1. **Phase 8.7**: Reject inference (teacher-student pipeline)
2. **Phase 8.8**: Full evaluation on D_R and comparison with all routes

---

*DCN-v2 Training Pipeline - Phase 8.6 - Pair-programming approved 2026-02-15*